# 01 — What is in the raw dataset

Read this before generating anything. It describes the BreastDCEDL release as it
arrives, and it establishes the one fact that constrains every design decision
downstream: the three cohorts are not interchangeable.

It writes nothing and trains nothing. It reads one CSV.

| | |
|---|---|
| source | [Zenodo 18114231](https://zenodo.org/records/18114231), MinCrop v4 |
| paper | [arXiv:2506.12190](https://arxiv.org/abs/2506.12190) |
| cohorts | I-SPY2 982, DUKE 916, I-SPY1 172 |

## Configuration

One path and the task definitions. Nothing below this cell hard-codes either.

In [ ]:
from pathlib import Path

# Repository root. The notebook lives in notebooks/, so the parent is the root.
REPO_ROOT = Path.cwd().parent

# INPUT, must exist. The raw Zenodo download. Never written to by anything in this
# project. Too large for version control, so it is not in the repository; get it
# from https://zenodo.org/records/18114231 (MinCrop v4).
RAW_DIR = REPO_ROOT / "raw_dataset_BreastDCEDL"

# INPUT, must exist. The authors' harmonised metadata: one row per patient with the
# labels, the official split, the voxel spacing, the tumour volume and the DUKE
# bounding boxes. Every label this project uses is read from here, never derived,
# so a label can never disagree with the published one.
METADATA_CSV = RAW_DIR / "BreastDCEDL_metadata_min_crop.csv"

# The four classification targets in the release. Each reads a different column and
# each column is missing for a different set of patients, which is what the
# availability table below is for.
TASKS = {
    "subtype": dict(column="HR_HER2_STATUS",
                    classes=("HRposHER2neg", "TripleNeg", "HER2pos"),
                    note="3 classes. This thesis's target. The authors never attempted it."),
    "her2":    dict(column="HER2", classes=("HER2neg", "HER2pos"),
                    note="binary. The authors report AUC 0.744 with THDA-ResNet."),
    "hr":      dict(column="HR", classes=("HRneg", "HRpos"),
                    note="binary. Positive in about 55% of I-SPY2."),
    "pcr":     dict(column="pCR", classes=("no_pCR", "pCR"),
                    note="binary. A PRE-treatment scan predicting a post-chemo outcome."),
}

# The official split, encoded as an integer in the `test` column.
SPLIT_MAP = {0: "train", 1: "test", 2: "val"}

assert METADATA_CSV.is_file(), (
    f"{METADATA_CSV} not found. Download the MinCrop release into {RAW_DIR}.")
print(f"metadata {METADATA_CSV}")

## Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({"figure.dpi": 120, "font.size": 9, "axes.grid": True,
                     "grid.alpha": 0.25, "axes.spines.top": False,
                     "axes.spines.right": False})

meta = pd.read_csv(METADATA_CSV)
print(f"{len(meta)} patients, {meta.shape[1]} metadata columns\n")
print(meta.dataset.value_counts().to_string())

## Label availability

Each task reads a different column and each column is missing for a different set of
patients. This table decides how many patients each experiment can use, and it is
why the thesis targets subtype rather than pCR: DUKE has a pCR label for only 298
of its 916 patients.

In [ ]:
rows = []
for name, spec in TASKS.items():
    for cohort, g in meta.groupby("dataset"):
        rows.append({"task": name, "column": spec["column"], "cohort": cohort,
                     "available": int(g[spec["column"]].notna().sum()),
                     "total": len(g)})
avail = pd.DataFrame(rows).pivot_table(index=["task", "column"], columns="cohort",
                                       values="available")
print(avail.to_string())
print()
for name, spec in TASKS.items():
    print(f"  {name:<8} {spec['note']}")

## The official split

The `test` column, mapped to names. Using the authors' split rather than inventing a
new one is what keeps any number this project produces comparable with the published
ones.

In [ ]:
m = meta.assign(split=meta.test.map(SPLIT_MAP))
print(pd.crosstab(m.dataset, m.split, margins=True).to_string())
print()
print(pd.crosstab(m.split, m.HR_HER2_STATUS, margins=True).to_string())

test_counts = m[m.split == "test"].HR_HER2_STATUS.value_counts()
print(f"\ntrivial baseline on the test split: {test_counts.max()}/{test_counts.sum()}"
      f" = {test_counts.max() / test_counts.sum():.4f}")
print("Accuracy below that number is worse than predicting one class for everybody.")

## The cohorts are not interchangeable

This is the single most important fact about this dataset.

A model trained on pooled cohorts can answer "which scanner produced this?" instead
of "which subtype is this?". Measured on exactly this data, a source probe, meaning
the identical pipeline with the cohort as the label, reaches patient-level macro AUC
**0.9978**. The actual subtype task reaches 0.6068.

The table below is why that shortcut pays. DUKE tumours are about five times smaller
by volume, and its majority class is 26 points more frequent than I-SPY2's. So
"small tumour, therefore DUKE, therefore HRposHER2neg" is a route to a respectable
score with no biology in it at all.

In [ ]:
m2 = meta[meta.HR_HER2_STATUS.notna()].copy()
# The largest side of the annotation box, in millimetres. The box is in pixels and
# xy_spacing varies by a factor of four across patients, so the pixel number alone
# would not be comparable between cohorts.
m2["box_side_px"] = np.maximum(m2.ecol - m2.scol, m2.eraw - m2.sraw)
m2["box_side_mm"] = m2.box_side_px * m2.xy_spacing

summary = m2.groupby("dataset").agg(
    patients=("pid", "size"),
    HRposHER2neg_pct=("HR_HER2_STATUS",
                      lambda s: round(100 * (s == "HRposHER2neg").mean(), 1)),
    tumour_mm_median=("box_side_mm", lambda s: round(s.median(), 1)),
    tumour_volume_median=("tum_vol", lambda s: round(s.median(), 2)),
    xy_spacing_median=("xy_spacing", lambda s: round(s.median(), 3)))
print(summary.to_string())

## The same thing as a picture

Three panels: the class prior per cohort, the tumour size distribution, and the
tumour volume on a log scale.

The class prior panel is the one that matters most. If the cohorts had the same
prior, identifying the cohort would buy the model nothing.

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(13, 3.6))
colours = {"spy2": "#2b6cb0", "duke": "#c05621", "spy1": "#2f855a"}

prop = pd.crosstab(m2.dataset, m2.HR_HER2_STATUS, normalize="index") * 100
prop[["HRposHER2neg", "TripleNeg", "HER2pos"]].plot(
    kind="barh", stacked=True, ax=ax[0], width=0.65,
    color=["#4c78a8", "#e45756", "#54a24b"])
ax[0].set_title("Class prior by cohort"); ax[0].set_xlabel("% of patients")
ax[0].set_ylabel(""); ax[0].legend(fontsize=7); ax[0].grid(False)

for c in colours:
    ax[1].hist(m2[m2.dataset == c].box_side_mm, bins=40, range=(0, 180), alpha=0.55,
               density=True, color=colours[c], label=c)
ax[1].set_title("Tumour size"); ax[1].set_xlabel("largest box side (mm)")
ax[1].legend(fontsize=7)

for c in colours:
    ax[2].hist(np.log10(m2[m2.dataset == c].tum_vol.clip(lower=0.01)), bins=40,
               alpha=0.55, density=True, color=colours[c], label=c)
ax[2].set_title("Tumour volume"); ax[2].set_xlabel("log10 volume")
ax[2].legend(fontsize=7)
plt.tight_layout(); plt.show()

## Voxel spacing, and why the crop is measured in millimetres

`xy_spacing` ranges from 0.31 to 1.41 mm per pixel across this release. A crop fixed
in pixels therefore covers a different amount of anatomy for every patient, and the
degree of magnification would itself identify the cohort.

That is the reasoning behind the 80 mm physical window in notebook 02. After
resizing to 224 pixels, every image in the generated dataset sits at the same 0.357
mm per pixel.

In [ ]:
print(f"xy_spacing across the release: {meta.xy_spacing.min():.3f} to "
      f"{meta.xy_spacing.max():.3f} mm per pixel\n")
print(meta.groupby("dataset").xy_spacing.describe()[
    ["min", "25%", "50%", "75%", "max"]].round(3).to_string())

CROP_MM, SAVE_SIZE = 80.0, 224
print(f"\nan {CROP_MM:.0f} mm window at each of those spacings:")
for q in (0.05, 0.5, 0.95):
    s = meta.xy_spacing.quantile(q)
    side = max(int(round(CROP_MM / s)), 8)
    print(f"  spacing {s:.3f} mm/px (quantile {q:.2f})  ->  {side:3d} px  "
          f"->  resized to {SAVE_SIZE} px = {s * side / SAVE_SIZE:.3f} mm/px")

## What this notebook established

Nothing was written. What it produced is four facts that everything downstream
depends on.

**The labels come from the release, never from code.** Every task reads one column
of the authors' harmonised metadata, so a label here cannot disagree with a
published one.

**The split comes from the release too.** 0 train, 1 test, 2 validation, used as
given.

**The trivial baseline on the pooled test split is 0.5112.** No accuracy in this
project means anything reported without it.

**The cohorts are separable, and separating them pays.** A source probe reaches
0.9978 against 0.6068 for the real task. The class priors differ by 26 points and
the tumour volumes by a factor of five.

That last one cuts both ways, and both sides matter to the thesis. It is a confound
that has to be reported beside every pooled result. It is also exactly what makes
the hospitals in notebook 06 genuinely non-IID rather than merely different in size,
which is what gives RQ2 something real to measure.

**Where this goes next.** Notebook 02 turns these volumes into the 2-D dataset.